# 9주차 딥러닝 실습 노트북  
## 당뇨 예측 데이터로 AI 신경망 직접 만들기

이 노트북의 목표는 **딥러닝의 작동 원리**를 코드로 직접 확인하는 것입니다.

오늘 실습 흐름은 다음과 같습니다.

1. 데이터 불러오기와 정규화  
2. 입력층·은닉층·출력층으로 AI 신경망 설계  
3. 손실함수와 학습률 설정  
4. 모델 학습 실행  
5. 손실 곡선과 정확도 곡선 확인  
6. 학습률을 바꿔보며 결과 비교  

> 핵심 문장: **딥러닝은 예측하고, 오차를 계산하고, 오차가 줄어드는 방향으로 가중치를 수정하는 과정을 반복하는 학습 방법입니다.**


---

## STEP 0. 실행 환경 준비

아래 셀은 필요한 라이브러리를 불러오고, 결과가 재현되도록 난수 시드를 고정합니다.

Colab에서 한글 그래프가 깨지는 경우를 줄이기 위해 **나눔고딕 폰트 설정**도 포함했습니다.


In [ ]:
# 필요한 라이브러리 설치 여부 확인
# Colab에는 대부분 기본 설치되어 있지만, 환경에 따라 tensorflow가 없을 수 있습니다.

try:
    import tensorflow as tf
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf

print("TensorFlow version:", tf.__version__)


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

from tensorflow import keras
from tensorflow.keras import layers

# 재현성을 위한 난수 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 한글 폰트 설정: Colab에서는 NanumGothic 설치 후 사용
try:
    if os.path.exists("/content"):
        # Colab 환경
        import subprocess, matplotlib.font_manager as fm
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False)
        font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
        if os.path.exists(font_path):
            fm.fontManager.addfont(font_path)
            plt.rcParams["font.family"] = "NanumGothic"
    else:
        # 로컬 환경: 설치된 한글 폰트 중 하나를 사용
        plt.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]
except Exception as e:
    print("폰트 설정 중 문제가 있었지만, 실습 진행에는 영향이 없습니다:", e)

plt.rcParams["axes.unicode_minus"] = False

print("실습 준비 완료!")


---

## STEP 1. 데이터 불러오기 + 정규화

이번 실습에서는 **Pima Indians Diabetes** 데이터를 사용합니다.

- 입력 특성: 임신횟수, 혈당, 혈압, 피부두께, 인슐린, BMI, 유전기능, 나이  
- 정답: 당뇨여부  
- 목표: 8개 입력 특성을 이용해 **당뇨 여부 0/1**을 예측합니다.

딥러닝 모델은 숫자의 단위를 자동으로 이해하지 못합니다.  
따라서 혈당, BMI, 나이처럼 범위가 다른 값들을 **정규화**해서 비슷한 스케일로 맞춥니다.


In [ ]:
# 데이터 불러오기
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"

columns = [
    "임신횟수", "혈당", "혈압", "피부두께",
    "인슐린", "BMI", "유전기능", "나이", "당뇨여부"
]

df = pd.read_csv(url, names=columns)

print("데이터 크기:", df.shape)
display(df.head())


In [ ]:
# 입력 X와 정답 y 분리
X = df.drop("당뇨여부", axis=1)
y = df["당뇨여부"]

# 훈련 데이터와 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("훈련 데이터:", X_train.shape)
print("테스트 데이터:", X_test.shape)
print("훈련 정답 비율")
print(y_train.value_counts(normalize=True).rename("비율"))


In [ ]:
# 정규화 전 예시 확인
sample_before = X_train.iloc[[0]].copy()

# StandardScaler: 평균 0, 표준편차 1로 변환
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 정규화 후 예시 확인
sample_after = pd.DataFrame(
    X_train_scaled[[0]],
    columns=X_train.columns
)

print("정규화 전")
display(sample_before)

print("정규화 후")
display(sample_after)


### 정규화가 필요한 이유

예를 들어 혈당은 100 이상, BMI는 20~40 정도, 나이는 20~80 정도로 범위가 다릅니다.  
AI는 처음에 이 숫자의 단위와 의미를 모릅니다. 숫자 범위가 큰 변수를 더 중요하게 오해할 수 있습니다.

정규화는 모든 입력값을 비슷한 기준으로 맞춰서, 모델이 더 안정적으로 학습하도록 도와줍니다.


---

## STEP 2. AI 뇌 설계하기

신경망은 크게 세 부분으로 볼 수 있습니다.

- **입력층**: 8개 특성을 받음  
- **은닉층**: 입력값 사이의 패턴을 학습  
- **출력층**: 당뇨일 확률을 0~1 사이 값으로 출력  

이번 모델은 다음 구조로 만듭니다.

```text
입력 8개 → Dense(16, ReLU) → Dense(8, ReLU) → Dense(1, Sigmoid)
```

- ReLU: 은닉층에서 주로 사용  
- Sigmoid: 이진 분류 출력층에서 사용


In [ ]:
def build_model(hidden1=16, hidden2=8, learning_rate=0.01):
    """
    당뇨 여부를 예측하는 간단한 딥러닝 모델을 생성합니다.

    hidden1: 첫 번째 은닉층 뉴런 수
    hidden2: 두 번째 은닉층 뉴런 수
    learning_rate: 학습률
    """
    model = keras.Sequential([
        layers.Input(shape=(X_train_scaled.shape[1],)),
        layers.Dense(hidden1, activation="relu"),
        layers.Dense(hidden2, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

model = build_model(hidden1=16, hidden2=8, learning_rate=0.01)
model.summary()


### 모델 구조 해석

- `Dense(16, activation="relu")`  
  8개 입력 특성을 받아 16개의 뉴런으로 처리합니다.

- `Dense(8, activation="relu")`  
  앞 층에서 나온 정보를 다시 조합해 더 복잡한 패턴을 학습합니다.

- `Dense(1, activation="sigmoid")`  
  최종적으로 당뇨일 확률을 0~1 사이 값으로 출력합니다.


---

## STEP 3. 손실함수와 학습률 이해하기

모델을 학습시키려면 두 가지를 정해야 합니다.

1. **손실함수**  
   모델이 얼마나 틀렸는지 계산하는 함수입니다.  
   당뇨 여부는 0/1 이진 분류 문제이므로 `binary_crossentropy`를 사용합니다.

2. **학습률**  
   가중치를 한 번에 얼마나 크게 수정할지 정하는 값입니다.  
   이번 기본값은 `0.01`입니다.


In [ ]:
# compile은 build_model 함수 안에서 이미 수행되었습니다.
# 여기서는 설정값만 다시 확인합니다.

print("손실함수:", model.loss)
print("옵티마이저:", type(model.optimizer).__name__)
print("학습률:", float(model.optimizer.learning_rate.numpy()))


---

## STEP 4. 모델 학습 실행

이제 모델을 실제로 학습시킵니다.

- `epochs=100`: 전체 훈련 데이터를 100번 반복 학습  
- `batch_size=32`: 데이터를 32개씩 묶어서 업데이트  
- `validation_split=0.2`: 훈련 데이터 중 20%를 검증용으로 사용  

학습 중에는 `loss`가 점점 내려가는지 확인합니다.


In [ ]:
EPOCHS = 100
BATCH_SIZE = 32

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1
)


---

## STEP 5. 손실 곡선과 정확도 곡선 확인

학습이 잘 되면 보통 에포크가 늘어날수록 손실이 내려갑니다.

그래프에서 볼 것:

1. 훈련 손실이 전체적으로 내려가는가?  
2. 검증 손실도 같이 내려가는가?  
3. 훈련 손실과 검증 손실의 차이가 너무 커지지는 않는가?  

훈련 손실은 내려가는데 검증 손실이 올라가면 **과적합**을 의심할 수 있습니다.


In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df.tail())

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="훈련 손실")
plt.plot(history_df["val_loss"], label="검증 손실")
plt.xlabel("에포크")
plt.ylabel("손실")
plt.title("손실 곡선")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="훈련 정확도")
plt.plot(history_df["val_accuracy"], label="검증 정확도")
plt.xlabel("에포크")
plt.ylabel("정확도")
plt.title("정확도 곡선")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 테스트 데이터에서 최종 성능 평가
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"테스트 손실: {test_loss:.4f}")
print(f"테스트 정확도: {test_acc:.4f}")

# 예측 확률과 최종 클래스
y_prob = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\n혼동행렬")
print(confusion_matrix(y_test, y_pred))

print("\n분류 리포트")
print(classification_report(y_test, y_pred, digits=3))


### 예측값 해석하기

Sigmoid 출력층은 0~1 사이의 값을 냅니다.

- 0.80 → 당뇨일 확률이 높다고 예측  
- 0.20 → 당뇨일 확률이 낮다고 예측  
- 일반적으로 0.5 이상이면 1, 0.5 미만이면 0으로 분류할 수 있습니다.


In [ ]:
# 테스트 데이터 일부 예측 결과 확인
result_df = X_test.copy().reset_index(drop=True)
result_df["실제_당뇨여부"] = y_test.reset_index(drop=True)
result_df["예측_당뇨확률"] = y_prob
result_df["예측_당뇨여부"] = y_pred

display(result_df.head(10))


---

## STEP 6. 학습률 3가지 실험

이번에는 학습률을 바꿔보겠습니다.

- `0.1`: 너무 클 수 있음 → 손실이 불안정하거나 발산할 가능성  
- `0.01`: 적절한 예시  
- `0.001`: 안정적이지만 느릴 수 있음  

같은 모델이라도 학습률에 따라 손실 곡선이 달라집니다.


In [ ]:
def train_with_learning_rate(lr, epochs=50):
    tf.random.set_seed(SEED)
    temp_model = build_model(hidden1=16, hidden2=8, learning_rate=lr)

    temp_history = temp_model.fit(
        X_train_scaled,
        y_train,
        epochs=epochs,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )

    return pd.DataFrame(temp_history.history)

learning_rates = [0.1, 0.01, 0.001]
lr_histories = {}

for lr in learning_rates:
    print(f"학습률 {lr} 실험 중...")
    lr_histories[lr] = train_with_learning_rate(lr, epochs=50)
    final_loss = lr_histories[lr]["loss"].iloc[-1]
    final_val_loss = lr_histories[lr]["val_loss"].iloc[-1]
    print(f"  최종 훈련 손실: {final_loss:.4f} | 최종 검증 손실: {final_val_loss:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))

for lr, hdf in lr_histories.items():
    plt.plot(hdf["loss"], label=f"lr={lr}")

plt.xlabel("에포크")
plt.ylabel("훈련 손실")
plt.title("학습률별 손실 곡선 비교")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

summary_rows = []
for lr, hdf in lr_histories.items():
    summary_rows.append({
        "학습률": lr,
        "최종_훈련손실": hdf["loss"].iloc[-1],
        "최종_검증손실": hdf["val_loss"].iloc[-1],
        "최종_훈련정확도": hdf["accuracy"].iloc[-1],
        "최종_검증정확도": hdf["val_accuracy"].iloc[-1],
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


### 학습률 실험 해석

그래프를 보면서 다음 질문에 답해보세요.

1. 어떤 학습률에서 손실이 가장 안정적으로 내려가나요?  
2. 너무 큰 학습률은 어떤 모습을 보이나요?  
3. 너무 작은 학습률은 왜 느리게 보이나요?  
4. 검증 손실과 훈련 손실의 차이가 큰 경우는 없나요?

> 실습 결과는 실행 환경과 난수 초기화에 따라 조금 달라질 수 있습니다.  
> 중요한 것은 “학습률이 바뀌면 학습 곡선도 달라진다”는 원리를 이해하는 것입니다.


---

## 추가 실습 1. 은닉층 뉴런 수 바꿔보기

아래 코드를 수정해서 은닉층 뉴런 수를 바꿔보세요.

추천 실험:

1. `hidden1=8, hidden2=4`  
2. `hidden1=16, hidden2=8`  
3. `hidden1=32, hidden2=16`  
4. `hidden1=64, hidden2=32`  

뉴런 수가 많아지면 더 복잡한 패턴을 학습할 수 있지만, 데이터가 충분하지 않으면 과적합될 수 있습니다.


In [ ]:
# 은닉층 뉴런 수 직접 바꿔보기
tf.random.set_seed(SEED)

custom_model = build_model(
    hidden1=32,
    hidden2=16,
    learning_rate=0.01
)

custom_history = custom_model.fit(
    X_train_scaled,
    y_train,
    epochs=80,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

custom_history_df = pd.DataFrame(custom_history.history)

plt.figure(figsize=(8, 5))
plt.plot(custom_history_df["loss"], label="훈련 손실")
plt.plot(custom_history_df["val_loss"], label="검증 손실")
plt.xlabel("에포크")
plt.ylabel("손실")
plt.title("은닉층 뉴런 수 변경 실험")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

custom_loss, custom_acc = custom_model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"테스트 손실: {custom_loss:.4f}")
print(f"테스트 정확도: {custom_acc:.4f}")


---

## 마무리 정리

오늘 실습에서 확인한 내용은 다음과 같습니다.

1. **정규화**  
   입력값의 범위를 맞춰야 모델이 안정적으로 학습합니다.

2. **신경망 구조**  
   입력층, 은닉층, 출력층으로 구성됩니다.

3. **손실함수**  
   예측이 얼마나 틀렸는지 숫자로 계산합니다.

4. **경사하강법과 학습률**  
   손실이 줄어드는 방향으로 가중치를 조금씩 수정합니다.

5. **손실 곡선**  
   AI가 제대로 배우는지 확인하는 가장 직관적인 도구입니다.

핵심 문장으로 마무리하겠습니다.

> 딥러닝은 처음부터 똑똑한 것이 아니라,  
> **예측하고, 틀리고, 오차를 계산하고, 가중치를 수정하는 과정을 반복하면서 점점 좋아지는 모델**입니다.
